# 02 — GRPO Reinforcement Learning

Trains GRPO Model

## 1. Setup

In [1]:
# Clone the repo
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Install dependencies
!pip install "protobuf<5" --quiet
!pip install -e . --quiet
!pip install bitsandbytes latex2sympy2 --quiet
!pip install vllm==0.12.0 --quiet
!pip install mergekit --quiet

# mergekit pulls in llm_blender which is broken with transformers>=4.45
# (TRANSFORMERS_CACHE removed). We don't use it, so uninstall.
!pip uninstall llm-blender -y --quiet 2>/dev/null; true

Cloning into 'math-rl-tuning'...
remote: Enumerating objects: 903, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 903 (delta 131), reused 153 (delta 67), pack-reused 681 (from 1)
Receiving objects: 100% (903/903), 4.66 MiB | 20.58 MiB/s, done.
Resolving deltas: 100% (560/560), done.
/content/math-rl-tuning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you hav

## 2. Config

In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
#import importlib.util

from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

cfg = load_config()


setup_hf_token()
setup_wandb(cfg.grpo_training.report_to and "math-rl-grpo")

mount_google_drive()

SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"
GRPO_CHECKPOINT = None

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Mounted at /content/drive


## 3. Run GRPO Training

In [3]:
# Delete old merged model cache (important after SFT config changes).
# merge_adapter() skips the merge if the output directory already exists —
# so if we changed LoRA config or re-ran SFT, we must clear this cache
# to force a fresh merge with the correct weights.
!rm -rf outputs/sft_merged

from math_rl_tuning.grpo_trainer import run_grpo_training

#runs GRPO, returns trainer, model, tokenizer, and reward callback for evaluation and inference after training
trainer, model, tokenizer, reward_callback = run_grpo_training(
    cfg,
    sft_adapter_path=SFT_ADAPTER_PATH,
    save_to_drive=True,
    checkpoint_path=GRPO_CHECKPOINT,
)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:446: UserWarning: Unmerge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


[step 51] boxed_format_reward_func: 0.4688 | correctness_reward_func: -0.2500
[step 52] boxed_format_reward_func: 0.5000 | correctness_reward_func: -0.5000
[step 53] boxed_format_reward_func: 0.4688 | correctness_reward_func: 0.8750
[step 54] boxed_format_reward_func: 0.4062 | correctness_reward_func: 0.6250
[step 55] boxed_format_reward_func: 0.3438 | correctness_reward_func: -0.7500
[step 56] boxed_format_reward_func: 0.3438 | correctness_reward_func: -0.6250
[step 57] boxed_format_reward_func: 0.4062 | correctness_reward_func: 0.3750
[step 58] boxed_format_reward_func: 0.5000 | correctness_reward_func: 0.5000
[step 59] boxed_format_reward_func: 0.5000 | correctness_reward_func: 0.1250
[step 60] boxed_format_reward_func: 0.5000 | correctness_reward_func: 0.8750
[step 61] boxed_format_reward_func: 0.5000 | correctness_reward_func: 0.5000
[step 62] boxed_format_reward_func: 0.4062 | correctness_reward_func: -0.6250
[step 63] boxed_format_reward_func: 0.4375 | correctness_reward_func: -

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:446: UserWarning: Unmerge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


[step 101] boxed_format_reward_func: 0.4062 | correctness_reward_func: -0.5000
[step 102] boxed_format_reward_func: 0.4688 | correctness_reward_func: -0.7500
[step 103] boxed_format_reward_func: 0.4375 | correctness_reward_func: -0.8750


KeyboardInterrupt: 

In [4]:
!cp -r /content/math-rl-tuning/outputs/grpo/checkpoint-100 /content/drive/MyDrive/math-rl-tuning/grpo_checkpoint_100


## 4. Eval


In [5]:
#shows reward history from GRPO training
reward_callback.plot()

NameError: name 'reward_callback' is not defined

In [ ]:
#example of grpo model after training

from math_rl_tuning.inference import generate

questions = [
    "What is 15% of 240?",
    "Solve for x: 3x + 7 = 22",
    "A rectangle has length 12 cm and width 5 cm. What is its area?",
]

for q in questions:
    print(f"Q: {q}")
    response = generate(q, model, tokenizer)
    print(f"A: {response[:300]}")
    print("-" * 40)

## 5. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()